In [1]:
import pandas as pd
import numpy as np
import glob
import backtrader as bt
import matplotlib.pyplot as plt
import pandas_ta as ta

# Ensure plots are rendered inline in the notebook
%matplotlib inline

In [2]:
# Define the path to your CSV files
file_path_pattern = '/root/github/CTI_Scripts/py_scripts/backtesting/historical_data/USDCAD/DAT_MT_USDCAD_M1_*.csv'

# Define the column names
column_names = ['DATE', 'TIME', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'TICKVOL', 'VOL', 'SPREAD']

# Load and combine all CSV files
all_files = glob.glob(file_path_pattern)
data_list = []

for file in all_files:
    print(f"Loading file: {file}")  # Debugging: Print file being loaded
    df = pd.read_csv(file, delimiter=',', names=column_names, header=None, dtype=str)
    # print(df.head())  # Debugging: Print the first few rows of the loaded DataFrame
    data_list.append(df)

combined_data = pd.concat(data_list)

# Check the combined data before parsing dates
print("Combined data before parsing dates:")
print(combined_data.head())

# Combine <DATE> and <TIME> into a single datetime column
combined_data['datetime'] = pd.to_datetime(combined_data['DATE'] + ' ' + combined_data['TIME'], format='%Y.%m.%d %H:%M', errors='coerce')

# Check the combined data after parsing dates
print("Combined data after parsing dates:")
print(combined_data.head())

# Check for rows with NaT in datetime column
print("Rows with NaT in datetime column:")
print(combined_data[combined_data['datetime'].isna()].head())

# Drop rows with NaT in datetime column
combined_data.dropna(subset=['datetime'], inplace=True)

# Set datetime as the index
combined_data.set_index('datetime', inplace=True)
combined_data.sort_index(inplace=True)

# Check the combined data before converting numeric columns
print("Combined data before converting numeric columns:")
print(combined_data.head())

# Convert numeric columns to appropriate data types
numeric_columns = ['OPEN', 'HIGH', 'LOW', 'CLOSE', 'TICKVOL', 'VOL', 'SPREAD']
combined_data[numeric_columns] = combined_data[numeric_columns].apply(pd.to_numeric, errors='coerce')

# Check the combined data after converting numeric columns
print("Combined data after converting numeric columns:")
print(combined_data.head())

# Drop rows with NaN values in numeric columns
combined_data.replace([np.inf, -np.inf], np.nan, inplace=True)
combined_data.fillna(0, inplace=True)

# Display the final combined data
print("Final combined data:")
combined_data.head()

Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/USDCAD/DAT_MT_USDCAD_M1_2016.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/USDCAD/DAT_MT_USDCAD_M1_2012.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/USDCAD/DAT_MT_USDCAD_M1_20220103_20230228.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/USDCAD/DAT_MT_USDCAD_M1_2018.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/USDCAD/DAT_MT_USDCAD_M1_2020.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/USDCAD/DAT_MT_USDCAD_M1_2021.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/USDCAD/DAT_MT_USDCAD_M1_2019.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/USDCAD/DAT_MT_USDCAD_M1_2014.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/USDCAD/DAT_MT_USDCAD_

,DATE,TIME,OPEN,HIGH,LOW,CLOSE,TICKVOL,VOL,SPREAD
datetime,,,,,,,,,
2012-01-02 02:00:00,2012.01.02,02:00,1.01926,1.01926,1.01925,1.01926,0,0.0,0.0
2012-01-02 02:04:00,2012.01.02,02:04,1.01925,1.01925,1.01925,1.01925,0,0.0,0.0
2012-01-02 02:05:00,2012.01.02,02:05,1.01918,1.01918,1.01917,1.01917,0,0.0,0.0
2012-01-02 02:06:00,2012.01.02,02:06,1.01918,1.01918,1.01918,1.01918,0,0.0,0.0
2012-01-02 02:08:00,2012.01.02,02:08,1.01925,1.01925,1.01925,1.01925,0,0.0,0.0


In [3]:


# Calculate indicators
combined_data['SMA_50'] = ta.sma(combined_data['CLOSE'], length=50)
combined_data['RSI'] = ta.rsi(combined_data['CLOSE'], length=14)

# Drop rows with NaN values (due to indicator calculation)
combined_data.dropna(inplace=True)

# Display the data with indicators
print("Data with indicators:")
combined_data.tail()

Data with indicators:


,DATE,TIME,OPEN,HIGH,LOW,CLOSE,TICKVOL,VOL,SPREAD,SMA_50,RSI
datetime,,,,,,,,,,,
2021-12-31 16:54:00,2021.12.31,16:54,1.26401,1.26422,1.26401,1.26422,0,0.0,0.0,1.265052,33.425622
2021-12-31 16:55:00,2021.12.31,16:55,1.26421,1.26434,1.26397,1.26412,0,0.0,0.0,1.265034,31.999885
2021-12-31 16:56:00,2021.12.31,16:56,1.26412,1.26419,1.26412,1.26418,0,0.0,0.0,1.265018,33.823771
2021-12-31 16:57:00,2021.12.31,16:57,1.26418,1.26419,1.26389,1.26391,0,0.0,0.0,1.264993,29.933001
2021-12-31 16:58:00,2021.12.31,16:58,1.26375,1.26404,1.26353,1.26377,0,0.0,0.0,1.264967,28.126343


In [4]:
# Define a custom strategy
class PandasTAStrategy(bt.Strategy):
    def __init__(self):
        self.sma_50 = bt.indicators.SimpleMovingAverage(self.data.close, period=50)
        self.rsi = bt.indicators.RelativeStrengthIndex(self.data.close, period=14)

    def next(self):
        if self.data.close[0] > self.sma_50[0] and self.rsi[0] < 30 and not self.position:
            self.buy()
        elif self.data.close[0] < self.sma_50[0] and self.rsi[0] > 70 and self.position:
            self.sell()

# Convert the pandas DataFrame to a Backtrader data feed
class PandasData(bt.feeds.PandasData):
    lines = ('sma_50', 'rsi',)
    params = (
        ('datetime', None),
        ('open', 'OPEN'),
        ('high', 'HIGH'),
        ('low', 'LOW'),
        ('close', 'CLOSE'),
        ('volume', 'VOL'),
        ('openinterest', None),
        ('sma_50', None),
        ('rsi', None),
    )

data_feed = PandasData(dataname=combined_data)

# Set up the Backtrader environment
cerebro = bt.Cerebro()
cerebro.addstrategy(PandasTAStrategy)
cerebro.adddata(data_feed)
cerebro.broker.set_cash(2500)
cerebro.broker.setcommission(commission=0.02)

# Run the backtest
cerebro.run()
print('Final Portfolio Value: %.2f' % cerebro.broker.getvalue())
# Plot the results
cerebro.plot()
plt.show()

Final Portfolio Value: 2500.00


<IPython.core.display.Javascript object>